# 03 — Stage 2 Coder fine-tune

QLoRA fine-tune of `Qwen2.5-Coder-7B-Instruct` on the `(pseudocode ->
python_code)` pairs in `data/stage2_coder/pseudotopython.jsonl`.

**Unlike notebook 02, there is no DSL special-token registration here.** Only
the Planner has to *emit* `<PLAN>`/`<STEP>`/`</PLAN>` as single tokens; the
Coder just reads them as ordinary text inside the pseudocode it's given, so
this notebook's tokenizer and base model are used completely stock — no
`add_special_tokens`, no `trainable_token_indices`, no vocab-size juggling.
That's also why this notebook has one fewer numbered section than notebook
02.

The system prompt used both for training (below) and for generation is
`CODER_SYSTEM_PROMPT` in `src/ui/generate.py` — imported, not duplicated, so
this notebook and `src/ui/console.py`'s coder side can never train and infer
with different prompts.

**This notebook is self-contained: `Runtime -> Run all` on a GPU runtime is
the whole procedure.** Section 2 clones the repo, mounts Drive, and checks
the GPU for you; there are no cells to add by hand. Each section says what
you should see in its output, so you can tell a good run from a bad one as
you go.

Two things worth doing *before* you connect to a runtime, because compute
units are billed on GPU-connected wall time:

1. Edit the section 1 config cell. Colab lets you edit cell source with no
   runtime attached — only *running* needs one.
2. `File -> Save a copy in Drive`, so your edits survive. Opened from the
   GitHub link, this notebook is a read-only render.

`docs/colab_setup.md` has the rest: choosing a GPU, what persists between
sessions, troubleshooting, and how to reuse the adapter afterwards — the
same file notebook 02 uses, since none of that changes for Stage 2.

## 1. Config

Everything you might want to change lives here. Paths are absolute and
derived from `REPO_DIR`, so nothing depends on the runtime's working
directory.

| Setting | Default | Notes |
| --- | --- | --- |
| `REPO_BRANCH` | `main` | Change if the data you want isn't merged yet. |
| `OUTPUT_DIR` | `stage2_coder_qlora` | Runtime-local, so epoch checkpoints die with the session. Point it inside `DRIVE_CHECKPOINT_DIR` if you expect interruptions and want to resume with `trainer.train(resume_from_checkpoint=True)`. |
| `CACHE_MODEL_ON_DRIVE` | `False` | `True` keeps the ~15GB base model on Drive so later sessions skip the download. Only worth it if your Drive has the space to spare. |
| `MAX_SEQ_LEN` | 1024 | Prompt + code, truncated. Raised from notebook 02's 512: reference `python_code` completions run up to ~2900 characters, longer than any Stage 1 pseudocode plan. |
| `NUM_EPOCHS` | 15 | High on purpose — 50 examples is a proof of concept, same as Stage 1. |
| `LEARNING_RATE` | 2e-4 | Standard LoRA range (1e-4 to 3e-4). |
| `PER_DEVICE_BATCH_SIZE` × `GRAD_ACCUM_STEPS` | 4 × 4 | Effective batch 16, ~3 steps per epoch. |
| `SPLIT_SEED` | 0 | Seeds the train/eval split. Shared with Stage 1 and the eval notebook — don't change it for one stage only. |
| `EVAL_FRACTION` | 0.2 | 50 examples → 40 train / 10 held out. |
| `EVAL_BATCH_SIZE` | 4 | Eval-only forward passes; no gradients, so it can exceed the train batch size. |

If you hit CUDA out of memory later, drop `PER_DEVICE_BATCH_SIZE` to 2 or 1
and raise `GRAD_ACCUM_STEPS` to keep the effective batch at 16.

In [ ]:
# Where the code and data come from
REPO_URL = "https://github.com/ashfordreyes/pythonllm.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/pythonllm"

DATA_PATH = f"{REPO_DIR}/data/stage2_coder/pseudotopython.jsonl"

# Where results go. Only Drive survives the runtime shutting down.
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/pythonllm_checkpoints"
OUTPUT_DIR = "stage2_coder_qlora"
CACHE_MODEL_ON_DRIVE = False

# Training
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 15
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4

# Held-out split. Both stages split by row index from this seed, so the same
# tasks are held out for Stage 1, Stage 2, and end-to-end eval. Changing the
# seed silently changes which examples the model has already seen.
SPLIT_SEED = 0
EVAL_FRACTION = 0.2
EVAL_BATCH_SIZE = 4

## 2. Setup

Four cells that turn a bare runtime into one this notebook can train on:
check the GPU, install the libraries, mount Drive, clone the repo. Run them
in order — the Drive mount has to happen before anything imports
`transformers` for `CACHE_MODEL_ON_DRIVE` to take effect.

Once these have run clean you don't touch them again for the session.

### 2a. Check the GPU

Colab silently falls back to whatever hardware is free, so confirm you got
what you asked for. Expect ~40GB total memory on an A100 or ~23GB on an L4,
and `True` for both torch checks.

`is_bf16_supported()` must be `True`: section 6 trains with `bf16=True`. A
free T4 reports `False` here — see `docs/colab_setup.md` for why that GPU
isn't practical for this run.

In [ ]:
!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())  # must be True

### 2b. Install dependencies

`bitsandbytes` provides the 4-bit (NF4) quantization and `peft` provides
LoRA. Takes 1-2 minutes.

If pip reports that it upgraded an already-imported package (usually `torch`
or `transformers`), restart the runtime and re-run from section 1 —
otherwise you get version-mismatch errors several cells later.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl matplotlib

### 2c. Mount Google Drive

**Nothing under `/content` survives the session** — not the clone, not the
model download, not `OUTPUT_DIR`. Drive is the only durable storage Colab
gives you, so this cell mounts it and creates the checkpoint folder that
section 8 copies the trained adapter into.

The first run opens a Google auth popup; approve it. Re-running when already
mounted is harmless.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

if CACHE_MODEL_ON_DRIVE:
    # Must be set before transformers is imported, or it is ignored.
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    print("HF cache on Drive:", os.environ["HF_HOME"])

print("Checkpoints will go to:", DRIVE_CHECKPOINT_DIR)

### 2d. Clone the repo

Opening this notebook does *not* bring the repo with it. The cells below
read the dataset off the runtime's disk and import `CODER_SYSTEM_PROMPT`
from `src/ui/generate.py`, `is_executable_example`/`split_dataset` from
`src/eval/scoring.py` and `src/splits.py`, so the files have to be there.

`ashfordreyes/pythonllm` is public, so no token is needed. The cell is safe
to re-run: it pulls instead of cloning if the directory already exists.

Expect `OK: 50 examples`. If the assert fires instead, the clone failed or
`REPO_BRANCH` names a branch that doesn't have the data — fix that before
going on, because the next section is where a missing clone would otherwise
surface as `ModuleNotFoundError: No module named 'ui'`.

In [ ]:
import pathlib
import sys

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already present; pulling")
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

assert pathlib.Path(DATA_PATH).exists(), (
    f"missing {DATA_PATH} — the clone failed, or branch '{REPO_BRANCH}' "
    "doesn't have this file"
)

sys.path.insert(0, f"{REPO_DIR}/src")

n_examples = sum(1 for _ in open(DATA_PATH))
print(f"OK: {n_examples} examples in {DATA_PATH}")

## 3. Load the tokenizer and base model in 4-bit

No special tokens to register (see the intro), so the tokenizer loads
straight from `BASE_MODEL` and the embedding matrix is left completely
untouched — unlike notebook 02, there's no resize question at all here,
because nothing is being added to the vocabulary.

Downloads ~15GB (3-10 minutes; the download is bf16, only the in-memory
weights are 4-bit).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)
model.config.use_cache = False

print(f"{BASE_MODEL} loaded in 4-bit; tokenizer vocab size {len(tokenizer)}")

## 4. LoRA setup

Same rank/alpha/dropout/target-modules as notebook 02: rank 16, alpha 32,
dropout 0.05, applied to all attention and MLP projections. No
`trainable_token_indices` this time — there are no new token rows to teach,
so `peft` only ever trains the LoRA matrices themselves.

Expect a trainable-parameter count in the same ballpark as notebook 02's
~40M (~0.5%): the two base models are both 7B-class with similar hidden
sizes, and notebook 02's number was already almost entirely LoRA matrices —
its ~21K token-embedding rows were a rounding error next to that.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Load and format the dataset

Same 40/10 split as Stage 1, derived the same way: `is_executable_example`
flags which reference snippets are safe to `exec()`, and that set is passed
to `split_dataset` as `priority` so a proportional share of them lands in
eval instead of leaving it to chance (`src/splits.py`'s docstring covers why
that's not "peeking" — it's a static property of the reference code, checked
before any split is drawn).

Notebook 02 computed this same priority set by reading this file as a side
table; here it *is* the file being trained on, so the priority set and the
dataset come from the same read. Same `SPLIT_SEED`/`EVAL_FRACTION` as
notebook 02 (passed explicitly here rather than re-imported from
`splits.py`, so an edit to this cell's config values above isn't silently
shadowed by that module's own defaults) reproduces the identical held-out
rows both other notebooks already use — no split file needed.

Each example is formatted with the base model's own chat template, using
`CODER_SYSTEM_PROMPT` from `src/ui/generate.py` as the system turn and the
`pseudocode` field as the user turn, then loss-masked so only the
`python_code` completion contributes to the loss.

Expect `50 examples -> 40 train / 10 eval`.

In [ ]:
from datasets import load_dataset

from eval.scoring import is_executable_example
from splits import read_jsonl, split_dataset

raw_dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# Same priority stratification notebook 02 computes from a side read of this
# same file -- read directly here since this file is what's being trained on.
stage2_rows = read_jsonl(DATA_PATH)
executable_priority = {
    i for i, row in enumerate(stage2_rows) if is_executable_example(row["python_code"])
}

train_raw, eval_raw = split_dataset(raw_dataset, SPLIT_SEED, EVAL_FRACTION, priority=executable_priority)

print(f"{len(raw_dataset)} examples -> {len(train_raw)} train / {len(eval_raw)} eval")
print("\nheld-out task 0 pseudocode:", eval_raw[0]["pseudocode"][:120])

In [ ]:
from ui.generate import CODER_SYSTEM_PROMPT


def format_example(example):
    prompt_messages = [
        {"role": "system", "content": CODER_SYSTEM_PROMPT},
        {"role": "user", "content": example["pseudocode"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    full_text = prompt_text + example["python_code"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

    labels = list(full["input_ids"])
    prompt_len = min(len(prompt_ids), len(labels))
    for i in range(prompt_len):
        labels[i] = -100

    full["labels"] = labels
    return full


train_dataset = train_raw.map(format_example, remove_columns=train_raw.column_names)
eval_dataset = eval_raw.map(format_example, remove_columns=eval_raw.column_names)

# Sanity check the masking: this should print the code and nothing else.
row = train_dataset[0]
supervised = [t for t, l in zip(row["input_ids"], row["labels"]) if l != -100]
print(f"{len(supervised)} of {len(row['labels'])} tokens supervised")
print(tokenizer.decode(supervised))

## 6. Train

15 epochs over 40 training examples is ~45 steps — a few minutes on an A100
or L4. Training loss is logged every step and should fall; eval loss on the
10 held-out examples is logged once per epoch.

Watch the gap between them. Training loss falling while eval loss flattens
or rises is memorization, and the fix is fewer epochs — not a better
checkpoint, which is why `load_best_model_at_end` is deliberately off.

If it stays flat or goes `nan`: flat usually means an example got fully
masked (check the supervised-token count printed above is not 0 — if it is,
the completion likely got truncated by `MAX_SEQ_LEN`, more plausible here
than in Stage 1 given how much longer the code completions run), and `nan`
usually means bf16 isn't really supported — see the section 2a output.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

data_collator = DataCollatorForSeq2Seq(
    tokenizer, padding=True, label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    bf16=True,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    # No load_best_model_at_end on purpose: picking a checkpoint by the loss
    # of 10 examples selects on noise. Read the curve in section 6b instead.
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()

## 6b. Loss curve

`trainer.state.log_history` only lives in memory, so dump it before anything
else can end the session. The JSON is the record; the PNG is for reading at a
glance.

Train and eval loss share one axis on purpose — they measure the same thing,
and the gap between them is the whole point. Eval loss climbing while train
loss keeps falling means the run is memorizing 40 examples; that is the
signal to cut `NUM_EPOCHS`.

In [ ]:
import json

from eval.plots import plot_loss_curve

LOSS_HISTORY_PATH = f"{OUTPUT_DIR}/log_history.json"
LOSS_PLOT_PATH = f"{OUTPUT_DIR}/loss_curve.png"

with open(LOSS_HISTORY_PATH, "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

plot_loss_curve(trainer.state.log_history, LOSS_PLOT_PATH, title="Stage 2 training loss")
print(f"Wrote {LOSS_HISTORY_PATH} and {LOSS_PLOT_PATH}")

eval_losses = [e["eval_loss"] for e in trainer.state.log_history if "eval_loss" in e]
if eval_losses:
    print(f"eval loss: {eval_losses[0]:.4f} (first epoch) -> {eval_losses[-1]:.4f} (last)")

from IPython.display import Image, display
display(Image(LOSS_PLOT_PATH))

## 7. Save the adapter and tokenizer

Both, together, for the same reason as notebook 02 even though this
tokenizer carries no extra tokens: `04_eval_pipeline.ipynb` and
`src/ui/generate.py` should load the tokenizer that trained alongside this
adapter rather than assuming a fresh one from the hub matches it.

Expect `adapter_model.safetensors` to be roughly 160-200MB — a bare LoRA
adapter, since section 3 never touched `embed_tokens` or `lm_head`.

In [ ]:
FINAL_DIR = f"{OUTPUT_DIR}/final"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Saved LoRA adapter + tokenizer to {FINAL_DIR}")

## 8. Copy the adapter to Google Drive

**Do not skip this.** `FINAL_DIR` is on the runtime's disk, which is
destroyed when the session ends, disconnects, or is reclaimed for idling.
This is the one artifact of the whole run you can't cheaply reproduce.

The loss history and its PNG go along with it: a checkpoint with no record
of how it trained is much harder to compare against the next one.

Saved to `stage2_coder`, parallel to notebook 02's `stage1_planner` —
`04_eval_pipeline.ipynb` section 7 expects the adapter at this path once it
exists. Drive is already mounted from section 2c.

In [ ]:
DRIVE_ADAPTER_DIR = f"{DRIVE_CHECKPOINT_DIR}/stage2_coder"

!rm -rf {DRIVE_ADAPTER_DIR}
!cp -r {FINAL_DIR} {DRIVE_ADAPTER_DIR}
# The loss curve and its raw log travel with the adapter -- without them
# there's no record of how this checkpoint trained.
!cp {LOSS_HISTORY_PATH} {LOSS_PLOT_PATH} {DRIVE_ADAPTER_DIR}/
!ls {DRIVE_ADAPTER_DIR}
print(f"Adapter, loss history and loss curve saved to {DRIVE_ADAPTER_DIR}")

## 9. Quick sanity check

Runs one **held-out** pseudocode plan through the fine-tuned model — a task
the model never trained on, so the output is a (very small) quality signal
rather than a recall test. The reference code is printed above it to
compare against.

This calls `ui.generate.make_coder` directly rather than hand-rolling
another `generate()` call, so this cell exercises the exact code path
`src/ui/console.py` uses — the same system prompt (imported, not
duplicated, in section 5), the same greedy defaults, the same fence
stripping. If this cell looks right, the console will too.

In [ ]:
from ui.generate import make_coder

model.eval()
coder = make_coder(model, tokenizer, stream=False, max_new_tokens=MAX_SEQ_LEN)

test_pseudocode = eval_raw[0]["pseudocode"]  # held out: the model never saw this
generated_code = coder(test_pseudocode)

print("REFERENCE:\n", eval_raw[0]["python_code"])
print("\nPSEUDOCODE INPUT:\n", test_pseudocode)
print("\nGENERATED CODE:\n", generated_code)

## 10. Release the runtime

Compute units are billed on GPU-connected wall time, and closing the browser
tab does **not** disconnect — Colab keeps the runtime alive in the
background, still billing. This cell ends the session and frees the GPU.

It kills the kernel, so only run it once section 8 has confirmed the adapter
is on Drive. Comment it out if you still want to poke at the in-memory
model.

In [ ]:
from google.colab import runtime

runtime.unassign()